##Import Libraries & Configure Environment

In [27]:
# ============================================================
# GENOMIC PREPROCESSING PIPELINE
# Breast Cancer Recurrence Prediction
# Master's Thesis
# ============================================================

import os
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("=" * 60)
print("Environment Ready")
print("=" * 60)

print(f"Numpy Version : {np.__version__}")
print(f"Pandas Version: {pd.__version__}")

Environment Ready
Numpy Version : 2.0.2
Pandas Version: 2.2.2


In [28]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Mount Google Drive & Define Paths

In [29]:
# ============================================================
# Define Dataset Paths
# ============================================================

BASE_DIR = "/content/drive/MyDrive/BreastCanser-DataSet"

GENE_FILE = os.path.join(
    BASE_DIR,
    "Human__TCGA_BRCA__UNC__RNAseq__HiSeq_RNA__01_28_2016__BI__Gene__Firehose_RSEM_log2.tsv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Genomic_Preprocessing"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Gene file:")
print(GENE_FILE)

print("\nOutput folder:")
print(OUTPUT_DIR)

Gene file:
/content/drive/MyDrive/BreastCanser-GenomicData/Human__TCGA_BRCA__UNC__RNAseq__HiSeq_RNA__01_28_2016__BI__Gene__Firehose_RSEM_log2.tsv

Output folder:
/content/drive/MyDrive/BreastCanser-GenomicData/Genomic_Preprocessing


##Load Dataset & Initial Inspection

In [30]:
# ============================================================
# Load RNA-Seq Dataset
# ============================================================

gene_df = pd.read_csv(
    GENE_FILE,
    sep="\t"
)

print("=" * 60)
print("RAW DATASET")
print("=" * 60)

print(f"Shape: {gene_df.shape}")

print("\nFirst five rows:")
display(gene_df.head())

print("\nColumn names (first 10):")
print(gene_df.columns[:10].tolist())



# ============================================================
# Dataset Summary
# ============================================================

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

num_genes = gene_df.shape[0]
num_patients = gene_df.shape[1] - 1

print(f"Genes (rows)      : {num_genes:,}")
print(f"Patients (columns): {num_patients:,}")

print("\nGene column:")
print(gene_df.columns[0])

print("\nExample patient IDs:")
print(gene_df.columns[1:11].tolist())

print("\nMissing values in dataset:")
print(gene_df.isna().sum().sum())

print("\nDuplicate gene names:")
print(gene_df.iloc[:,0].duplicated().sum())

print("\nDuplicate patient columns:")
print(pd.Index(gene_df.columns[1:]).duplicated().sum())

RAW DATASET
Shape: (20155, 1094)

First five rows:


,attrib_name,TCGA.3C.AAAU,TCGA.3C.AALI,TCGA.3C.AALJ,TCGA.3C.AALK,TCGA.4H.AAAK,TCGA.5L.AAT0,TCGA.5L.AAT1,TCGA.5T.A9QA,TCGA.A1.A0SB,...,TCGA.UL.AAZ6,TCGA.UU.A93S,TCGA.V7.A7HQ,TCGA.W8.A86G,TCGA.WT.AB41,TCGA.WT.AB44,TCGA.XX.A899,TCGA.XX.A89A,TCGA.Z7.A8R5,TCGA.Z7.A8R6
0,A1BG,7.6300,7.8971,8.7287,7.5851,8.0762,7.6779,7.7299,8.3001,5.6496,...,5.4968,8.3353,10.0127,9.0608,6.9166,8.8835,7.8088,8.0008,8.7831,7.9619
1,A1CF,0.0000,0.0000,0.9310,0.0000,0.5115,0.0000,0.0000,0.6069,0.0000,...,0.4846,0.0000,0.0000,0.5603,0.0000,0.0000,0.0000,1.7492,0.0000,0.0000
2,A2BP1,0.0000,0.0000,0.0000,0.0000,2.2720,0.6659,0.0000,0.6069,2.4607,...,0.0000,0.0000,0.6922,2.6371,2.5190,0.0000,0.4789,1.2733,0.6756,0.0000
3,A2LD1,6.6999,6.1672,7.3422,5.9888,7.2796,6.8142,6.6861,7.8742,5.2816,...,7.5688,7.8208,6.9794,5.9240,8.2903,5.9711,7.0469,6.3386,6.3628,4.7108
4,A2ML1,1.2501,2.4196,0.0000,1.4087,2.1389,1.1198,2.3434,0.0000,2.0543,...,7.1771,0.0000,4.0885,1.2773,0.0000,2.9100,0.4789,5.8263,1.7610,2.3718



Column names (first 10):
['attrib_name', 'TCGA.3C.AAAU', 'TCGA.3C.AALI', 'TCGA.3C.AALJ', 'TCGA.3C.AALK', 'TCGA.4H.AAAK', 'TCGA.5L.AAT0', 'TCGA.5L.AAT1', 'TCGA.5T.A9QA', 'TCGA.A1.A0SB']
DATASET SUMMARY
Genes (rows)      : 20,155
Patients (columns): 1,093

Gene column:
attrib_name

Example patient IDs:
['TCGA.3C.AAAU', 'TCGA.3C.AALI', 'TCGA.3C.AALJ', 'TCGA.3C.AALK', 'TCGA.4H.AAAK', 'TCGA.5L.AAT0', 'TCGA.5L.AAT1', 'TCGA.5T.A9QA', 'TCGA.A1.A0SB', 'TCGA.A1.A0SD']

Missing values in dataset:
0

Duplicate gene names:
0

Duplicate patient columns:
0


##Convert Patient IDs

In [31]:
# ============================================================
# Convert Patient IDs
# TCGA.A1.A0SB -> TCGA-A1-A0SB
# ============================================================

gene_df.columns = [
    gene_df.columns[0]
] + [
    c.replace(".", "-")
    for c in gene_df.columns[1:]
]

print("="*60)
print("Patient IDs Converted")
print("="*60)

print(gene_df.columns[:10])



Patient IDs Converted
Index(['attrib_name', 'TCGA-3C-AAAU', 'TCGA-3C-AALI', 'TCGA-3C-AALJ',
       'TCGA-3C-AALK', 'TCGA-4H-AAAK', 'TCGA-5L-AAT0', 'TCGA-5L-AAT1',
       'TCGA-5T-A9QA', 'TCGA-A1-A0SB'],
      dtype='object')


##Convert Gene Values to Numeric

In [32]:
# ============================================================
# Convert Gene Values to Numeric
# ============================================================

gene_df.iloc[:,1:] = gene_df.iloc[:,1:].apply(
    pd.to_numeric,
    errors="coerce"
)

print("="*60)
print("Numeric Conversion Finished")
print("="*60)

print(gene_df.dtypes.head())

Numeric Conversion Finished
attrib_name      object
TCGA-3C-AAAU    float64
TCGA-3C-AALI    float64
TCGA-3C-AALJ    float64
TCGA-3C-AALK    float64
dtype: object


##Remove Zero Variance Genes

In [33]:
# ============================================================
# Remove Zero Variance Genes
# ============================================================

print("="*60)
print("ZERO VARIANCE FILTER")
print("="*60)


# Calculate variance for each gene across all patients

gene_variance = gene_df.iloc[:,1:].var(axis=1)


# Keep genes with variance > 0

non_zero_variance = gene_variance > 0


gene_filtered = gene_df[
    non_zero_variance
].copy()


gene_filtered = gene_filtered.reset_index(drop=True)


print("Original genes :", len(gene_df))
print("Remaining genes:", len(gene_filtered))
print("Removed genes  :", len(gene_df)-len(gene_filtered))

ZERO VARIANCE FILTER
Original genes : 20155
Remaining genes: 20155
Removed genes  : 0


##Remove Near-Zero Variance Genes

In [34]:
# ============================================================
# Remove Near-Zero Variance Genes
# ============================================================

gene_variance = gene_filtered.iloc[:,1:].var(axis=1)

threshold = 0.01

gene_filtered = gene_filtered[
    gene_variance > threshold
].reset_index(drop=True)

print("="*60)
print("NEAR ZERO VARIANCE FILTER")
print("="*60)

print("Remaining genes:",len(gene_filtered))

NEAR ZERO VARIANCE FILTER
Remaining genes: 19835


##Rank Genes by Variance

In [35]:
# ============================================================
# Rank Genes by Variance
# ============================================================

gene_filtered["Variance"] = (
    gene_filtered.iloc[:,1:]
    .var(axis=1)
)

gene_filtered = gene_filtered.sort_values(
    "Variance",
    ascending=False
)

print("="*60)
print("Highest Variance Genes")
print("="*60)

display(
    gene_filtered[
        ["attrib_name","Variance"]
    ].head(20)
)

Highest Variance Genes


,attrib_name,Variance
3821,CLEC3A,26.966259
4112,CPB1,26.713156
15207,SCGB2A2,25.293741
15204,SCGB1D2,20.070605
17266,TFF1,19.709392
7391,GSTM1,18.720623
13118,PIP,18.121910
15071,S100A7,17.674703
11173,MUCL1,17.670668
4507,CYP2B7P1,17.401513


##Select Top 1000 Genes

In [36]:
# ============================================================
# Select Top 1000 Genes
# ============================================================

TOP_GENES = 1000

gene_top = gene_filtered.head(TOP_GENES).copy()

print("="*60)
print("TOP VARIABLE GENES")
print("="*60)

print("Selected genes:",len(gene_top))

TOP VARIABLE GENES
Selected genes: 1000


##Save Selected Gene List

In [37]:
# ============================================================
# Save Gene List
# ============================================================

gene_top[["attrib_name","Variance"]].to_csv(

    os.path.join(
        OUTPUT_DIR,
        "selected_gene_list.csv"
    ),

    index=False
)

print("Gene list saved.")

Gene list saved.


##Create Patient-Level Genomic Matrix (Transpose)

In [38]:
# ============================================================
# Create Patient-Level Genomic Matrix
# Genes -> Columns
# Patients -> Rows
# ============================================================


import pandas as pd
import os


print("="*70)
print("CREATING PATIENT-LEVEL GENOMIC MATRIX")
print("="*70)


# Remove variance column if exists
if "Variance" in gene_top.columns:
    gene_matrix = gene_top.drop(
        columns=["Variance"]
    )
else:
    gene_matrix = gene_top.copy()


# Set gene names as index

gene_matrix = gene_matrix.set_index(
    "attrib_name"
)


# Transpose

genomic_patient = gene_matrix.T


# Rename index

genomic_patient.index.name = "patient_id"


print("Patients :", genomic_patient.shape[0])
print("Genes    :", genomic_patient.shape[1])


print("\nFirst rows:")
display(genomic_patient.head())

CREATING PATIENT-LEVEL GENOMIC MATRIX
Patients : 1093
Genes    : 1000

First rows:


attrib_name,CLEC3A,CPB1,SCGB2A2,SCGB1D2,TFF1,GSTM1,PIP,S100A7,MUCL1,CYP2B7P1,...,DNAH5,DMKN,SLC1A6,PTGDS,TFCP2L1,ZIC5,CRYAB,EMILIN3,TNNT2,INA
patient_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU,2.4543,16.1613,15.9692,12.2314,6.3268,10.8344,13.0549,0.000,8.3070,9.7704,...,8.1773,6.6771,0.4273,6.7059,9.5730,0.4273,6.4739,2.8300,5.6322,1.4454
TCGA-3C-AALI,11.7687,4.2437,16.1618,10.5341,7.5467,6.5291,8.3734,0.000,7.3052,2.6865,...,7.1481,3.9710,3.1065,8.3567,3.8673,5.1399,7.2288,3.1974,6.7710,1.0618
TCGA-3C-AALJ,0.0000,11.1077,0.0000,0.9310,7.8249,2.8770,0.0000,0.000,11.3664,9.9113,...,6.0306,0.0000,0.0000,6.8255,5.5054,4.1143,9.1020,4.1278,6.3032,1.4922
TCGA-3C-AALK,5.3766,6.9285,8.7442,6.5496,13.5956,11.6371,11.5943,6.214,12.0994,9.1587,...,7.1961,7.8827,0.0000,8.5023,6.6181,0.0000,8.7161,3.8877,3.6054,0.4995
TCGA-4H-AAAK,7.4930,2.7075,14.9834,9.3356,12.3546,11.3659,7.6139,0.000,7.2726,9.6403,...,8.6912,8.6382,0.0000,7.4410,9.2873,0.0000,8.7379,4.3130,1.4341,1.4341


##StandardScaler Normalization

In [39]:
# ============================================================
# Standardization
# ============================================================

from sklearn.preprocessing import StandardScaler


print("="*70)
print("STANDARDIZATION")
print("="*70)


scaler = StandardScaler()


scaled_values = scaler.fit_transform(
    genomic_patient
)


genomic_scaled = pd.DataFrame(
    scaled_values,
    index=genomic_patient.index,
    columns=genomic_patient.columns
)


print("Shape:")
print(genomic_scaled.shape)


print("\nMean:")
print(genomic_scaled.mean().mean())


print("\nStd:")
print(genomic_scaled.std().mean())

STANDARDIZATION
Shape:
(1093, 1000)

Mean:
-4.551399117614226e-17

Std:
1.0004577706808773


##Quality Control

In [40]:
# ============================================================
# QUALITY CONTROL
# ============================================================


print("="*70)
print("GENOMIC QUALITY CONTROL")
print("="*70)


qc = {}


qc["patients"] = genomic_scaled.shape[0]

qc["genes"] = genomic_scaled.shape[1]


qc["missing_values"] = (
    genomic_scaled.isna()
    .sum()
    .sum()
)


qc["infinite_values"] = (
    (~genomic_scaled.replace(
        [float("inf"), float("-inf")],
        pd.NA
    ).notna())
    .sum()
    .sum()
)


qc["duplicate_patients"] = (
    genomic_scaled.index.duplicated()
    .sum()
)


qc["duplicate_genes"] = (
    genomic_scaled.columns.duplicated()
    .sum()
)



for k,v in qc.items():
    print(f"{k}: {v}")


print("\nValue distribution:")
display(
    genomic_scaled.describe()
)

GENOMIC QUALITY CONTROL
patients: 1093
genes: 1000
missing_values: 0
infinite_values: 0
duplicate_patients: 0
duplicate_genes: 0

Value distribution:


attrib_name,CLEC3A,CPB1,SCGB2A2,SCGB1D2,TFF1,GSTM1,PIP,S100A7,MUCL1,CYP2B7P1,...,DNAH5,DMKN,SLC1A6,PTGDS,TFCP2L1,ZIC5,CRYAB,EMILIN3,TNNT2,INA
count,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,...,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03,1.093000e+03
mean,-6.338327e-16,1.461066e-15,1.015758e-15,1.733289e-15,4.098785e-15,6.760882e-16,2.642595e-15,4.810628e-16,1.950255e-15,-2.678350e-15,...,-1.602459e-15,-6.105922e-15,-3.607971e-16,-8.727389e-16,3.685981e-15,1.495195e-16,-1.898248e-15,-1.059638e-15,-9.426230e-16,9.044305e-16
std,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,...,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00,1.000458e+00
min,-9.178925e-01,-1.267645e+00,-1.891635e+00,-1.636181e+00,-1.916251e+00,-1.298324e+00,-1.973497e+00,-8.530970e-01,-1.712813e+00,-1.869744e+00,...,-3.600830e+00,-3.726541e+00,-5.808467e-01,-3.590751e+00,-3.428517e+00,-5.997344e-01,-2.315402e+00,-1.698349e+00,-1.043650e+00,-8.790223e-01
25%,-9.178925e-01,-8.394311e-01,-7.472521e-01,-8.098245e-01,-6.670097e-01,-9.657040e-01,-7.132956e-01,-8.530970e-01,-7.295261e-01,-9.426050e-01,...,-4.901390e-01,-4.639973e-01,-5.808467e-01,-6.104348e-01,-7.046167e-01,-5.997344e-01,-7.199982e-01,-7.015546e-01,-8.279823e-01,-6.283451e-01
50%,-4.611182e-01,-1.446340e-01,4.694128e-02,-1.902028e-02,2.647548e-01,-1.411223e-02,1.179782e-01,-4.560674e-01,-9.968968e-02,1.745996e-01,...,1.941841e-01,1.642518e-01,-3.974506e-01,8.355133e-03,2.435746e-02,-5.997344e-01,-1.288497e-01,-1.676008e-01,-2.722322e-01,-2.959168e-01
75%,8.124130e-01,6.025623e-01,8.439793e-01,7.429316e-01,7.731723e-01,1.013761e+00,7.587249e-01,6.213906e-01,6.712078e-01,8.145406e-01,...,7.063030e-01,5.682263e-01,6.498216e-02,6.938830e-01,6.746361e-01,1.036434e-01,5.695206e-01,4.917330e-01,6.014767e-01,1.743451e-01
max,2.508348e+00,2.789720e+00,2.281520e+00,2.825615e+00,1.991044e+00,2.260445e+00,2.774407e+00,3.199846e+00,2.840769e+00,2.053867e+00,...,2.225314e+00,3.944044e+00,5.232145e+00,4.057156e+00,2.881160e+00,3.633310e+00,3.440260e+00,4.087329e+00,4.762728e+00,4.920262e+00


##Save Final Genomic Dataset

In [41]:
# ============================================================
# SAVE FINAL OUTPUTS
# ============================================================


OUTPUT_DIR = "/content/drive/MyDrive/TCGA_BRCA/genomic_processed"


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# Save processed genomic matrix

genomic_scaled.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "genomic_processed.csv"
    )
)



# Save statistics

gene_statistics = pd.DataFrame({

    "gene":
    genomic_scaled.columns,

    "mean":
    genomic_scaled.mean(),

    "std":
    genomic_scaled.std(),

    "variance":
    genomic_scaled.var()

})


gene_statistics.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "gene_statistics.csv"
    ),
    index=False
)



# Save preprocessing report

with open(
    os.path.join(
        OUTPUT_DIR,
        "preprocessing_report.txt"
    ),
    "w"
) as f:

    f.write(
f"""
TCGA-BRCA RNA-seq preprocessing report

Input:
Firehose RNA-seq log2 normalized expression

Final dataset:

Patients:
{genomic_scaled.shape[0]}

Genes:
{genomic_scaled.shape[1]}


Preprocessing:

- TCGA patient IDs converted
- Duplicate genes checked
- Duplicate patients checked
- Missing values checked
- Zero variance genes removed
- Near-zero variance genes removed
- Top 1000 variable genes selected
- Data transposed to patient-level format
- StandardScaler normalization applied


Quality control:

Missing values:
{qc["missing_values"]}

Infinite values:
{qc["infinite_values"]}

Duplicate patients:
{qc["duplicate_patients"]}

Duplicate genes:
{qc["duplicate_genes"]}

"""
    )


print("="*70)
print("SAVED SUCCESSFULLY")
print("="*70)


print(
os.listdir(OUTPUT_DIR)
)

SAVED SUCCESSFULLY
['genomic_processed.csv', 'gene_statistics.csv', 'preprocessing_report.txt']
